# Knowledge Graph Parser
Extracts **(disease, symptom, importance)** tuples from `DerivedKnowledgeGraph_final.csv`
and writes them to `disease_symptom_importance.csv`.

In [1]:
import pandas as pd
import re

# ── 1. Load the source file ───────────────────────────────────────────────────
INPUT_FILE  = 'DerivedKnowledgeGraph_final.csv'   # adjust path if needed
OUTPUT_FILE = 'disease_symptom_importance.csv'

df_raw = pd.read_csv(INPUT_FILE)
print(f'Loaded {len(df_raw)} diseases.')
df_raw.head(3)

Loaded 156 diseases.


,Diseases,Symptoms
0,abscess,"pain (0.318), fever (0.119), swelling (0.112),..."
1,acid reflux,"pain (0.225), nausea (0.140), pain in upper ab..."
2,acute renal failure,"kidney failure (0.127), weakness (0.087), diar..."


In [2]:
# ── 2. Parse each row into (disease, symptom, importance) tuples ──────────────
#
# The Symptoms column holds a comma-separated list of tokens, each formatted as:
#   symptom name (importance_value)
# e.g.  'pain (0.318), fever (0.119), sore throat (0.021)'
#
# Strategy: split on ',' then match each trimmed token against the pattern
#   ^<name> (<number>)$
# This cleanly handles multi-word symptom names (e.g. 'sore throat').

TOKEN = re.compile(r'^(.+?)\s*\(([0-9.]+)\)$')

records = []
for _, row in df_raw.iterrows():
    disease  = str(row['Diseases']).strip()
    symptoms = str(row['Symptoms'])

    tokens = [t.strip() for t in symptoms.split(',')]
    for token in tokens:
        m = TOKEN.match(token)
        if m:
            symptom    = m.group(1).strip()
            importance = float(m.group(2))
            records.append({'disease': disease, 'symptom': symptom, 'importance': importance})

df_out = pd.DataFrame(records, columns=['disease', 'symptom', 'importance'])
print(f'Extracted {len(df_out):,} disease-symptom pairs.')
df_out.head(10)

Extracted 3,709 disease-symptom pairs.


,disease,symptom,importance
0,abscess,pain,0.318
1,abscess,fever,0.119
2,abscess,swelling,0.112
3,abscess,redness,0.094
4,abscess,chills,0.092
5,abscess,infection,0.083
6,abscess,cyst,0.047
7,abscess,tenderness,0.037
8,abscess,rectal pain,0.026
9,abscess,lesion,0.025


In [3]:
# ── 3. Quick sanity checks ────────────────────────────────────────────────────
print('Unique diseases :', df_out['disease'].nunique())
print('Unique symptoms :', df_out['symptom'].nunique())
print('Importance range:', df_out['importance'].min(), '-', df_out['importance'].max())
print()
print('Pairs per disease - distribution:')
print(df_out.groupby('disease').size().describe())

Unique diseases : 156
Unique symptoms : 330
Importance range: 0.002 - 0.881

Pairs per disease - distribution:
count    156.000000
mean      23.775641
std        7.949497
min       10.000000
25%       18.000000
50%       22.000000
75%       28.250000
max       54.000000
dtype: float64


In [4]:
# ── 4. Save to CSV ────────────────────────────────────────────────────────────
df_out.to_csv(OUTPUT_FILE, index=False)
print(f'Saved -> {OUTPUT_FILE}  ({len(df_out):,} rows)')

Saved -> disease_symptom_importance.csv  (3,709 rows)
